# Task 1 — Baseline System Effectiveness

Evaluate the face recognition system on **held-out images** of enrolled users (`data/enrolled_test/`).

### Protocol
- **Genuine pairs**: each test image matched against the correct enrolled user's embedding → label = 1
- **Impostor pairs**: each test image matched against a randomly chosen *different* enrolled user → label = 0  
  (sampled so that `n_impostor ≈ n_genuine` for a balanced evaluation)
- **Minimum**: ≥ 500 total probes required by project spec

### Metrics reported
| Metric | Description |
|--------|-------------|
| FAR | False Accept Rate — impostors accepted / all impostors |
| FRR | False Reject Rate — genuines rejected / all genuines |
| Accuracy | (TP + TN) / total |
| EER | Equal Error Rate — threshold where FAR ≈ FRR |
| AUC | Area under ROC curve |

## 1. Setup

In [ ]:
import sys
import random
import warnings
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import auc, roc_curve, confusion_matrix, ConfusionMatrixDisplay

sys.path.insert(0, str(Path('..').resolve()))

from src.model import get_insightface_model, get_embedding, cosine_similarity
from src.database import EmbeddingDB
from src.metrics import compute_far_frr, compute_roc, eer, plot_roc, plot_far_frr_vs_threshold
from src.utils import list_images

warnings.filterwarnings('ignore')

SEED             = 44
DEFAULT_THRESH   = 0.4
TEST_DIR         = Path('../data/enrolled_test')

random.seed(SEED)
np.random.seed(SEED)

print('Loading model...')
app = get_insightface_model('buffalo_l', ctx_id=0)
db  = EmbeddingDB.from_file()

enrolled_users = set(db.get_all_users())
print(f'Enrolled users in DB : {len(enrolled_users)}')
print(f'Test directory       : {TEST_DIR.resolve()}')

## 2. Build probe set

In [ ]:
# Collect all test images that belong to enrolled users
# Each entry: (user_id, embedding)

print('Extracting embeddings from test images...')

probes = []   # list of (user_id, embedding)

user_dirs = sorted(d for d in TEST_DIR.iterdir() if d.is_dir() and d.name in enrolled_users)

for user_dir in user_dirs:
    user_id = user_dir.name
    imgs = list_images(user_dir)
    for img_path in imgs:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        emb = get_embedding(app, img, fallback=True)
        if emb is None:
            continue
        probes.append((user_id, emb))

print(f'Total valid probes : {len(probes)}')
print(f'Users with probes  : {len({uid for uid, _ in probes})}')

if len(probes) < 500:
    print(f'WARNING: only {len(probes)} probes — spec requires >= 500. '
          'Check enrolled_test/ or run enroll.py to rebuild the split.')

## 3. Compute genuine and impostor scores

In [ ]:
user_list = sorted(enrolled_users)  # stable order for reproducible random sampling
rng = random.Random(SEED)

scores       = []
labels       = []
meta         = []   # (true_user_id, matched_user_id, pair_type)

for user_id, probe_emb in probes:
    # --- Genuine ---
    refs = db.get_user_embeddings(user_id)
    if refs:
        sim = max(float(np.dot(probe_emb, ref)) for ref in refs)
        scores.append(sim)
        labels.append(1)
        meta.append((user_id, user_id, 'genuine'))

    # --- Impostor (one random different user per probe) ---
    candidates = [u for u in user_list if u != user_id]
    if candidates:
        impostor_id = rng.choice(candidates)
        imp_refs = db.get_user_embeddings(impostor_id)
        if imp_refs:
            sim = max(float(np.dot(probe_emb, ref)) for ref in imp_refs)
            scores.append(sim)
            labels.append(0)
            meta.append((user_id, impostor_id, 'impostor'))

scores = np.array(scores)
labels = np.array(labels)

n_genuine  = int((labels == 1).sum())
n_impostor = int((labels == 0).sum())
print(f'Genuine  pairs : {n_genuine}')
print(f'Impostor pairs : {n_impostor}')
print(f'Total          : {len(scores)}')
print(f'Balance ratio  : {n_genuine / max(n_impostor, 1):.2f} (1.0 = perfect)')

## 4. Score distributions

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(scores[labels == 1], bins=60, alpha=0.65, color='steelblue',  label='Genuine')
ax.hist(scores[labels == 0], bins=60, alpha=0.65, color='tomato',     label='Impostor')
ax.axvline(DEFAULT_THRESH, color='k', ls='--', lw=1.5, label=f'Threshold = {DEFAULT_THRESH}')

ax.set_xlabel('Cosine similarity')
ax.set_ylabel('Count')
ax.set_title('Score distributions — Task 1 baseline')
ax.legend()
plt.tight_layout()

out_dir = Path('../results/task1')
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / 'score_distributions.png', dpi=150)
plt.show()

## 5. Metrics at default threshold

In [ ]:
far, frr, acc = compute_far_frr(scores, labels, DEFAULT_THRESH)

print(f'=== Metrics at threshold = {DEFAULT_THRESH} ===')
print(f'  FAR      : {far:.4f}  ({far*100:.2f}%)')
print(f'  FRR      : {frr:.4f}  ({frr*100:.2f}%)')
print(f'  Accuracy : {acc:.4f}  ({acc*100:.2f}%)')

## 6. ROC curve & EER

In [ ]:
fpr, tpr, thresholds = compute_roc(scores, labels)
eer_val, eer_thresh  = eer(fpr, tpr, thresholds)
auc_val              = auc(fpr, tpr)

print(f'AUC : {auc_val:.4f}')
print(f'EER : {eer_val:.4f}  ({eer_val*100:.2f}%)  at threshold = {eer_thresh:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
plot_roc(fpr, tpr, ax=axes[0], label='Task 1 Baseline')
axes[0].scatter([eer_val], [1 - eer_val], color='red', zorder=5, label=f'EER = {eer_val:.3f}')
axes[0].legend()

# FAR / FRR vs threshold
plot_far_frr_vs_threshold(scores, labels, ax=axes[1])
axes[1].axvline(DEFAULT_THRESH, color='k', ls='--', lw=1.2, label=f'Default ({DEFAULT_THRESH})')
axes[1].axvline(eer_thresh,     color='r', ls=':',  lw=1.2, label=f'EER ({eer_thresh:.3f})')
axes[1].legend()

plt.tight_layout()
plt.savefig(out_dir / 'roc_and_far_frr.png', dpi=150)
plt.show()

## 7. Confusion matrix at default threshold

In [ ]:
decisions = (scores >= DEFAULT_THRESH).astype(int)
cm = confusion_matrix(labels, decisions)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=['Impostor', 'Genuine'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion matrix  (threshold = {DEFAULT_THRESH})')
plt.tight_layout()
plt.savefig(out_dir / 'confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN (correct reject)  : {tn}')
print(f'FP (false accept)    : {fp}')
print(f'FN (false reject)    : {fn}')
print(f'TP (correct accept)  : {tp}')

## 8. Sweep of thresholds — summary table

In [ ]:
sweep_thresholds = np.arange(0.1, 0.9, 0.05)
rows = []
for t in sweep_thresholds:
    f, r, a = compute_far_frr(scores, labels, t)
    rows.append({'Threshold': round(float(t), 2), 'FAR': round(f, 4), 'FRR': round(r, 4), 'Accuracy': round(a, 4)})

df = pd.DataFrame(rows)
df_styled = df.style.highlight_min(subset=['FAR'], color='#c6efce') \
                    .highlight_min(subset=['FRR'], color='#ffeb9c') \
                    .highlight_max(subset=['Accuracy'], color='#dae8fc')
display(df_styled)

## 9. Per-user error analysis

Which users cause the most **False Rejects** (their genuine images are rejected) and which cause the most **False Accepts** (an impostor probe is mistakenly matched to them)?

In [ ]:
decisions = (scores >= DEFAULT_THRESH).astype(int)

# Build a DataFrame with one row per scored pair
df_err = pd.DataFrame(meta, columns=['true_user', 'matched_user', 'pair_type'])
df_err['score']    = scores
df_err['label']    = labels
df_err['decision'] = decisions
df_err['correct']  = (df_err['decision'] == df_err['label'])

# False Reject: genuine pair rejected (decision=0, label=1)
false_rejects = df_err[(df_err['label'] == 1) & (df_err['decision'] == 0)]

# False Accept: impostor pair accepted (decision=1, label=0)
false_accepts = df_err[(df_err['label'] == 0) & (df_err['decision'] == 1)]

print(f'False Rejects (FN) : {len(false_rejects)}')
print(f'False Accepts (FP) : {len(false_accepts)}')

In [ ]:
# ── Per-user False Reject Rate ────────────────────────────────────────────────
# For each user: how many of their genuine probes were incorrectly rejected?

genuine_df = df_err[df_err['label'] == 1].copy()
per_user_genuine = genuine_df.groupby('true_user').agg(
    n_probes   =('label', 'count'),
    n_rejected =('correct', lambda x: (~x).sum()),
).reset_index()
per_user_genuine['FRR'] = per_user_genuine['n_rejected'] / per_user_genuine['n_probes']
per_user_genuine = per_user_genuine.sort_values('FRR', ascending=False)

# ── Per-user False Accept Rate ────────────────────────────────────────────────
# For each user: how many impostor probes were wrongly matched to them?

impostor_df = df_err[df_err['label'] == 0].copy()
per_user_impostor = impostor_df.groupby('matched_user').agg(
    n_probes  =('label', 'count'),
    n_accepted=('correct', lambda x: (~x).sum()),
).reset_index()
per_user_impostor.columns = ['matched_user', 'n_impostor_probes', 'n_false_accepts']
per_user_impostor['FAR'] = per_user_impostor['n_false_accepts'] / per_user_impostor['n_impostor_probes']
per_user_impostor = per_user_impostor.sort_values('FAR', ascending=False)

# Show top offenders
TOP_N = 15

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# False Reject Rate per user (who gets rejected most often)
top_frr = per_user_genuine.head(TOP_N)
axes[0].barh(top_frr['true_user'], top_frr['FRR'], color='steelblue')
axes[0].axvline(frr, color='red', ls='--', lw=1.2, label=f'Mean FRR = {frr:.3f}')
axes[0].set_xlabel('FRR (False Reject Rate)')
axes[0].set_title(f'Top {TOP_N} users — highest False Reject Rate')
axes[0].invert_yaxis()
axes[0].legend()

# False Accept Rate per user (who acts as a "magnet" for impostors)
top_far = per_user_impostor.head(TOP_N)
axes[1].barh(top_far['matched_user'], top_far['FAR'], color='tomato')
axes[1].axvline(far, color='navy', ls='--', lw=1.2, label=f'Mean FAR = {far:.3f}')
axes[1].set_xlabel('FAR (False Accept Rate)')
axes[1].set_title(f'Top {TOP_N} users — highest False Accept Rate')
axes[1].invert_yaxis()
axes[1].legend()

plt.tight_layout()
plt.savefig(out_dir / 'per_user_errors.png', dpi=150)
plt.show()

In [ ]:
# Detailed tables — users with at least one error

print('=== Users with FALSE REJECTS (their images were rejected) ===')
frr_errors = per_user_genuine[per_user_genuine['n_rejected'] > 0].copy()
frr_errors['FRR'] = frr_errors['FRR'].map('{:.2%}'.format)
display(frr_errors.rename(columns={
    'true_user': 'User',
    'n_probes':  'Test images',
    'n_rejected':'Rejected',
}).reset_index(drop=True))

print(f'\n=== Users with FALSE ACCEPTS (impostors mistaken for them) ===')
far_errors = per_user_impostor[per_user_impostor['n_false_accepts'] > 0].copy()
far_errors['FAR'] = far_errors['FAR'].map('{:.2%}'.format)
display(far_errors.rename(columns={
    'matched_user':       'User (matched to)',
    'n_impostor_probes':  'Impostor probes',
    'n_false_accepts':    'Accepted',
}).reset_index(drop=True))

## 9. Summary

In [ ]:
far_def, frr_def, acc_def = compute_far_frr(scores, labels, DEFAULT_THRESH)

print('=' * 50)
print('TASK 1 — BASELINE RESULTS')
print('=' * 50)
print(f'  Enrolled users         : {len(enrolled_users)}')
print(f'  Test images (probes)   : {len(probes)}')
print(f'  Genuine  pairs         : {n_genuine}')
print(f'  Impostor pairs         : {n_impostor}')
print(f'  Total scored           : {len(scores)}')
print()
print(f'  Default threshold      : {DEFAULT_THRESH}')
print(f'  FAR  @ default         : {far_def*100:.2f}%')
print(f'  FRR  @ default         : {frr_def*100:.2f}%')
print(f'  Acc  @ default         : {acc_def*100:.2f}%')
print()
print(f'  EER                    : {eer_val*100:.2f}%')
print(f'  EER threshold          : {eer_thresh:.4f}')
print(f'  AUC                    : {auc_val:.4f}')
print('=' * 50)